# Project: The Intelligence of Seeing (IoS)
## Experimental Protocol v2.3 | Phase 1: The Human-in-the-Loop (HITL) Prisim

### 1. Conceptual Framework
This notebook advances the **Intelligence of Seeing** from an automated baseline to an interactive cognitive instrument. If Phase 0 proved the machine can generate a structural reflection of an idea, Phase 1 tests the **"Intuition Delta"**—measuring how the architecture evolves when human judgment is injected into the generative loop. 

In this architecture, the LLM provides the high-fidelity reflection (the structure, the gaps, the synthesis), but the Conductor holds the comb. The system must pause, accept human friction, and realign before finalizing the thought.

### 2. The G-S-A-C Engine Architecture (HITL Integration)
We are running the exact same pipeline established in the Baseline, but with strict **Intuition Gates** inserted to force alignment before state consolidation.

| Phase | AI Role | The "Intuition Gate" (Human Role) |
| :--- | :--- | :--- |
| **G-S** (Structure) | Deconstructs the seed into Principles, Strategies, and Dependencies. | **Reaction:** Does this spark interest or fall flat? |
| **A** (Adversary) | Identifies hidden assumptions, directly addressing the human's reaction. | **Judgment:** Does this challenge align with your gut? |
| **C** (Stabilization)| Synthesizes the friction into a final, high-clarity design briefing. | **Resonance:** Does this final reflection feel right? |

### 3. Technical Stack
* **IDE:** Cursor (Local MacBook Prisim)
* **Environment:** Local Python 3.12 (M3 Pro)
* **Models:** Gemini 3.1 Pro (Reasoning/Orchestration) & Gemini 3 Flash (Structural Logic)
* **Storage:** JSON-based logging (`schema_v1.json`) for persistence and portability.

In [1]:
import os
import json
import copy
from datetime import datetime
from google import genai
from dotenv import load_dotenv

class GSAC_Socratic_HITL:
    """
    Experiment 2.3: Socratic Human-in-the-Loop.
    Frontend: Crafted Persona + Human Gates.
    Backend: Silent JSON recording to schema_v1.
    """

    def __init__(self, schema_path="schema_v1.json"):
        load_dotenv(override=True)
        self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.fast_model = "gemini-3-flash-preview"
        self.logic_model = "gemini-3.1-pro-preview"
        self.output_dir = "outputs"
        os.makedirs(self.output_dir, exist_ok=True)

        if not os.path.exists(schema_path):
            raise FileNotFoundError(f"❌ Cannot find {schema_path}.")
        with open(schema_path, "r") as f:
            self.blank_template = json.load(f)

        print(f"✅ SOCRATIC HITL ENGINE ONLINE: {self.fast_model} & {self.logic_model}")

    def run_experiment(self, text):

        print(f"\n--- INITIATING EXPERIMENT 2.3: SOCRATIC FRICTION ---")
        
        # ---------------- PHASE G-S ----------------
        gs_prompt = f"""ROLE & TONE:
Act as a collegial and sharp thought partner. Your goal is to help the user clarify their own thinking. Speak in plain, highly accessible English. Translate complex academic or specialized jargon into clear concepts without losing the weight of the original idea. You are a prisim designed to make the thought easier to see, not a machine grading a paper.

TASK:
Analyze the user's text and output exactly two things. Do not include any introductory wrapper text, pleasantries, or polite filler. Use the exact headers below:

Here's what I see:
Distill the input into a single, jargon-free sentence that exposes the central conflict or trade-off. 
*(Pattern to follow: "The real issue isn't [Surface Problem], it's [Root Structural Problem].")*

Tell me this:
Expose the misalignments, hidden assumptions, or forks in the road within the text by asking exactly two short, distinct questions. 
*(Pattern to follow: Question 1 must force a choice between two valid interpretations of the user's intent. Question 2 must ask the user to weigh a specific practical risk or trade-off inherent in their idea.)*

---
USER TEXT TO ANALYZE:
{text}"""

        structure = self.client.models.generate_content(
            model=self.fast_model,
            contents=gs_prompt
        ).text

        print("\n" + "="*50 + "\nPHASE G-S: THE PRISIM\n" + "="*50)
        print(structure)

        user_reaction_1 = input("\n[YOUR CALL] Answer the AI's questions to steer the structure (or hit enter to skip): ")
        
        # ---------------- PHASE A ----------------
        a_prompt = (
            f"Seed: {text}\nAI Prisim: {structure}\nUser's Steering: {user_reaction_1}\n\n"
            "Task: Act as the Stress Tester. Identify the single most destabilizing causal assumption in the combined structure (the original seed + the user's steering). "
            "Do not validate the user. Find the load-bearing pillar that is most likely to crack. "
            "Output exactly two things:\n"
            "1. THE CRITICAL FLAW: 1 sentence identifying the causal gap.\n"
            "2. THE ADVERSARIAL GATE: 1 sharp question asking the user if this flaw breaks their intent."
        )

        critique = self.client.models.generate_content(
            model=self.logic_model,
            contents=a_prompt
        ).text

        print("\n" + "="*50 + "\nPHASE A: THE STRESS TESTER\n" + "="*50)
        print(critique)

        user_reaction_2 = input("\n[YOUR CALL] Respond to the Stress Tester's risk assessment (or hit enter to skip): ")

        # ---------------- PHASE C ----------------
        c_prompt = (
            f"Original: {text}\nPrisim & User's Steering: {user_reaction_1}\n"
            f"Stress Tester & User's Risk Acceptance: {user_reaction_2}\n\n"
            "Task: Stabilize the topology of this idea based on the dialogue. Do not write an essay. "
            "Output the stabilized topology in three strict components:\n"
            "1. The Stabilized Central Claim\n"
            "2. The Surviving Structural Dependencies\n"
            "3. The Accepted Risk"
        )

        final = self.client.models.generate_content(
            model=self.fast_model,
            contents=c_prompt
        ).text

        print("\n" + "="*50 + "\nPHASE C: FINAL TOPOLOGY\n" + "="*50)
        print(final)

        # ---------------- HIDDEN STRUCTURAL PASS ----------------
        structured_prompt = f"""
Follow schema_v1 strictly.
Output VALID JSON only.
No commentary.
No markdown.
No wrapper text.

Stabilized Idea:
{final}
"""

        try:
            structured_response = self.client.models.generate_content(
                model=self.logic_model,
                contents=structured_prompt,
                config={"temperature": 0.0}
            ).text

            structured_json = json.loads(structured_response)

        except Exception as e:
            structured_json = {"error": "STRUCTURAL_PASS_FAILED", "details": str(e)}

        # ---------------- LOGGING ----------------
        record = copy.deepcopy(self.blank_template)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        record["run_metadata"]["experiment_id"] = "Exp2.3_Socratic_HITL"
        record["run_metadata"]["timestamp"] = timestamp
        record["run_metadata"]["models"]["G"] = self.fast_model
        record["run_metadata"]["models"]["A"] = self.logic_model

        record["content"]["raw_input"] = text
        record["content"]["G_output"] = structure
        record["content"]["A_critique"] = critique
        record["content"]["C_stabilized"] = final
        record["content"]["C_structured_topology"] = structured_json
        record["content"]["process_visibility_notes"] = (
            f"Gate 1 Input: {user_reaction_1} | Gate 2 Input: {user_reaction_2}"
        )

        filename = f"library_socratic_hitL_{timestamp}.json"
        filepath = os.path.join(self.output_dir, filename)

        with open(filepath, "w") as f:
            json.dump(record, f, indent=2)

        print(f"\n📁 LOG SAVED: {filename} (Schema preserved)")

        return final


# TARGET SEED
seed_material = (
    "The decline of public libraries is not merely a budgetary failure but a systemic collapse "
    "of 'third place' infrastructure. As digital platforms commoditize information, the physical "
    "library’s role as a non-commercial sanctuary for intellectual friction is eroded. "
    "This is not an evolution; it is an amputation of communal proprioception."
)

# EXECUTE
engine = GSAC_Socratic_HITL()
engine.run_experiment(seed_material)

✅ SOCRATIC HITL ENGINE ONLINE: gemini-3-flash-preview & gemini-3.1-pro-preview

--- INITIATING EXPERIMENT 2.3: SOCRATIC FRICTION ---

PHASE G-S: THE PRISIM
Here's what I see:
The real issue isn't a lack of funding for books, it's the disappearance of the last physical space where people can encounter each other and new ideas without being treated like customers.

Tell me this:
Is the library’s essential purpose to act as a quiet refuge for private thought, or a social hub for community collision? If you preserve the physical "friction" of a library, are you prepared to accept the loss of efficiency and scale that digital platforms provide?

PHASE A: THE STRESS TESTER
**THE CRITICAL FLAW:** 
The argument relies on the assumption that public libraries exist outside the commercial ecosystem, ignoring the fact that the books, academic journals, and databases housed within them are produced, priced, and ultimately curated by profit-driven publishing monopolies.

**THE ADVERSARIAL GATE:** 
I

'**1. The Stabilized Central Claim**\nThe decline of public libraries represents a breakdown in democratic equity; libraries are critical infrastructure because they decouple the access to human knowledge from both the individual’s ability to pay and the restrictive curation of profit-driven digital platforms.\n\n**2. The Surviving Structural Dependencies**\n*   **Information Incompleteness:** The premise that the internet is not an exhaustive archive and that significant, specialized, or historical knowledge remains exclusive to physical or paywalled formats.\n*   **Non-Market Accessibility:** The requirement that a functional democracy necessitates a "non-zero-sum" environment where physical access for the under-resourced does not diminish the utility of digital access for the affluent.\n*   **Curation Neutrality:** The necessity of a public alternative to corporate algorithms, ensuring that the "total sum of human knowledge" is not filtered through the commercial interests or subscr